In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, round

In [3]:
spark = SparkSession.builder \
    .appName("Week6Assignment") \
    .getOrCreate()

In [4]:
path="/content/OnlineRetail.csv"

df=spark.read.csv(
    path,
    header=True,
    inferSchema=True
)

In [5]:
print("First 5 rows")

df.show(5)

print("Schema")

df.printSchema()

First 5 rows
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|   InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|12/1/2010 8:26|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|12/1/2010 8:26|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
only showing top 5 rows
Schema
root
 |-- InvoiceNo: string (

In [6]:
selected_df=df.select(
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "UnitPrice",
    "CustomerID",
    "Country"
)

selected_df.show(5)

+---------+---------+--------------------+--------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|     3.39|     17850|United Kingdom|
+---------+---------+--------------------+--------+---------+----------+--------------+
only showing top 5 rows


In [7]:
transformed_df=selected_df.withColumnRenamed(
    "Description",
    "Product_Name"
)

transformed_df.show(5)

+---------+---------+--------------------+--------+---------+----------+--------------+
|InvoiceNo|StockCode|        Product_Name|Quantity|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|     3.39|     17850|United Kingdom|
+---------+---------+--------------------+--------+---------+----------+--------------+
only showing top 5 rows


In [8]:
transformed_df=transformed_df.withColumn(
    "Quantity",
    col("Quantity").cast("integer")
)

transformed_df=transformed_df.withColumn(
    "UnitPrice",
    col("UnitPrice").cast("double")
)

transformed_df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)



In [9]:
transformed_df=transformed_df.withColumn(
    "TotalPrice",
    round(
        col("Quantity")*col("UnitPrice"),
        2
    )
)

transformed_df.show(5)

+---------+---------+--------------------+--------+---------+----------+--------------+----------+
|InvoiceNo|StockCode|        Product_Name|Quantity|UnitPrice|CustomerID|       Country|TotalPrice|
+---------+---------+--------------------+--------+---------+----------+--------------+----------+
|   536365|   85123A|WHITE HANGING HEA...|       6|     2.55|     17850|United Kingdom|      15.3|
|   536365|    71053| WHITE METAL LANTERN|       6|     3.39|     17850|United Kingdom|     20.34|
|   536365|   84406B|CREAM CUPID HEART...|       8|     2.75|     17850|United Kingdom|      22.0|
|   536365|   84029G|KNITTED UNION FLA...|       6|     3.39|     17850|United Kingdom|     20.34|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|     3.39|     17850|United Kingdom|     20.34|
+---------+---------+--------------------+--------+---------+----------+--------------+----------+
only showing top 5 rows


In [10]:
cleaned_df=transformed_df.dropna(
    subset=["CustomerID"]
)

cleaned_df.show(5)

+---------+---------+--------------------+--------+---------+----------+--------------+----------+
|InvoiceNo|StockCode|        Product_Name|Quantity|UnitPrice|CustomerID|       Country|TotalPrice|
+---------+---------+--------------------+--------+---------+----------+--------------+----------+
|   536365|   85123A|WHITE HANGING HEA...|       6|     2.55|     17850|United Kingdom|      15.3|
|   536365|    71053| WHITE METAL LANTERN|       6|     3.39|     17850|United Kingdom|     20.34|
|   536365|   84406B|CREAM CUPID HEART...|       8|     2.75|     17850|United Kingdom|      22.0|
|   536365|   84029G|KNITTED UNION FLA...|       6|     3.39|     17850|United Kingdom|     20.34|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|     3.39|     17850|United Kingdom|     20.34|
+---------+---------+--------------------+--------+---------+----------+--------------+----------+
only showing top 5 rows


In [11]:
filtered_df=cleaned_df.filter(
    (col("Quantity")>10) &
    (col("UnitPrice")>2)
)

filtered_df.show(5)

+---------+---------+--------------------+--------+---------+----------+-------+----------+
|InvoiceNo|StockCode|        Product_Name|Quantity|UnitPrice|CustomerID|Country|TotalPrice|
+---------+---------+--------------------+--------+---------+----------+-------+----------+
|   536370|    22728|ALARM CLOCK BAKEL...|      24|     3.75|     12583| France|      90.0|
|   536370|    22727|ALARM CLOCK BAKEL...|      24|     3.75|     12583| France|      90.0|
|   536370|    22726|ALARM CLOCK BAKEL...|      12|     3.75|     12583| France|      45.0|
|   536370|    21035|SET/2 RED RETROSP...|      18|     2.95|     12583| France|      53.1|
|   536370|    22326|ROUND SNACK BOXES...|      24|     2.95|     12583| France|      70.8|
+---------+---------+--------------------+--------+---------+----------+-------+----------+
only showing top 5 rows


In [12]:
country_df=cleaned_df.filter(
    (col("Country")=="United Kingdom") |
    (col("Quantity")>20)
)

country_df.show(5)

+---------+---------+--------------------+--------+---------+----------+--------------+----------+
|InvoiceNo|StockCode|        Product_Name|Quantity|UnitPrice|CustomerID|       Country|TotalPrice|
+---------+---------+--------------------+--------+---------+----------+--------------+----------+
|   536365|   85123A|WHITE HANGING HEA...|       6|     2.55|     17850|United Kingdom|      15.3|
|   536365|    71053| WHITE METAL LANTERN|       6|     3.39|     17850|United Kingdom|     20.34|
|   536365|   84406B|CREAM CUPID HEART...|       8|     2.75|     17850|United Kingdom|      22.0|
|   536365|   84029G|KNITTED UNION FLA...|       6|     3.39|     17850|United Kingdom|     20.34|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|     3.39|     17850|United Kingdom|     20.34|
+---------+---------+--------------------+--------+---------+----------+--------------+----------+
only showing top 5 rows


In [13]:
print("Total Rows")

print(filtered_df.count())

Total Rows
3462


In [14]:
filtered_df.write \
.mode("overwrite") \
.option("header",True) \
.csv("/content/output_csv")

In [15]:
filtered_df.write \
.mode("overwrite") \
.parquet("/content/output_parquet")

In [16]:
print("CSV and Parquet files saved successfully.")

CSV and Parquet files saved successfully.


In [17]:
csv_df = spark.read.csv("/content/output_csv", header=True, inferSchema=True)

csv_df.show(5)

+---------+---------+--------------------+--------+---------+----------+-------+----------+
|InvoiceNo|StockCode|        Product_Name|Quantity|UnitPrice|CustomerID|Country|TotalPrice|
+---------+---------+--------------------+--------+---------+----------+-------+----------+
|   536370|    22728|ALARM CLOCK BAKEL...|      24|     3.75|     12583| France|      90.0|
|   536370|    22727|ALARM CLOCK BAKEL...|      24|     3.75|     12583| France|      90.0|
|   536370|    22726|ALARM CLOCK BAKEL...|      12|     3.75|     12583| France|      45.0|
|   536370|    21035|SET/2 RED RETROSP...|      18|     2.95|     12583| France|      53.1|
|   536370|    22326|ROUND SNACK BOXES...|      24|     2.95|     12583| France|      70.8|
+---------+---------+--------------------+--------+---------+----------+-------+----------+
only showing top 5 rows


In [18]:
parquet_df = spark.read.parquet("/content/output_parquet")

parquet_df.show(5)

+---------+---------+--------------------+--------+---------+----------+-------+----------+
|InvoiceNo|StockCode|        Product_Name|Quantity|UnitPrice|CustomerID|Country|TotalPrice|
+---------+---------+--------------------+--------+---------+----------+-------+----------+
|   536370|    22728|ALARM CLOCK BAKEL...|      24|     3.75|     12583| France|      90.0|
|   536370|    22727|ALARM CLOCK BAKEL...|      24|     3.75|     12583| France|      90.0|
|   536370|    22726|ALARM CLOCK BAKEL...|      12|     3.75|     12583| France|      45.0|
|   536370|    21035|SET/2 RED RETROSP...|      18|     2.95|     12583| France|      53.1|
|   536370|    22326|ROUND SNACK BOXES...|      24|     2.95|     12583| France|      70.8|
+---------+---------+--------------------+--------+---------+----------+-------+----------+
only showing top 5 rows


In [19]:
!zip -r output_csv.zip /content/output_csv
!zip -r output_parquet.zip /content/output_parquet

  adding: content/output_csv/ (stored 0%)
  adding: content/output_csv/part-00001-a4f05db1-6838-4219-8d0c-914e13faf4a5-c000.csv (deflated 80%)
  adding: content/output_csv/part-00000-a4f05db1-6838-4219-8d0c-914e13faf4a5-c000.csv (deflated 82%)
  adding: content/output_csv/.part-00000-a4f05db1-6838-4219-8d0c-914e13faf4a5-c000.csv.crc (stored 0%)
  adding: content/output_csv/.part-00001-a4f05db1-6838-4219-8d0c-914e13faf4a5-c000.csv.crc (stored 0%)
  adding: content/output_csv/_SUCCESS (stored 0%)
  adding: content/output_csv/._SUCCESS.crc (stored 0%)
  adding: content/output_parquet/ (stored 0%)
  adding: content/output_parquet/.part-00000-4eadccb0-f4bd-4ee1-a45a-38c107bb6412-c000.snappy.parquet.crc (stored 0%)
  adding: content/output_parquet/part-00001-4eadccb0-f4bd-4ee1-a45a-38c107bb6412-c000.snappy.parquet (deflated 25%)
  adding: content/output_parquet/_SUCCESS (stored 0%)
  adding: content/output_parquet/.part-00001-4eadccb0-f4bd-4ee1-a45a-38c107bb6412-c000.snappy.parquet.crc (stor